# Huan luyen PiperPlus Tieng Viet (sea-g2p) tren Kaggle - Giong ngochuyen

Notebook nay thuc hien toan bo quy trinh huan luyen va xuat model ONNX cho bo du lieu don nguoi noi Tieng Viet:

- **WAV directory**: `/kaggle/input/datasets/fablenguyen2messi/ngochuyen1/wavs`
- **Metadata CSV**: `/kaggle/input/datasets/fablenguyen2messi/ngochuyen1/metadata.csv` (dinh dang `file.wav|ngochuyen|text`)
- **G2P Engine**: `sea-g2p` (`piper_plus_g2p.vietnamese`, 6 thanh dieu `vi_tone1..6`, PUA v3 `0xE065..0xE06C`, tong 199 symbols)

## Hai che do huan luyen (`TRAIN_MODE`)

1. **`finetune_zs_v1` (Khuyen nghi)**: Transfer learning tu `ayousanz/piper-plus-base` (`releases/zs-v1`, 173 symbols, 6 ngon ngu) sang 199 symbols va 7 ngon ngu (`vi = 6`). Su dung `--expand-embeddings`, `--no-freeze-dp`, va `--prosody-language-ids 0,6` de Duration Predictor hoc nhip dieu va thanh dieu Tieng Viet nhanh hon va tu nhien hon khi du lieu it.
2. **`scratch`**: Huan luyen moi hoan toan tu epoch 0 voi 1 ngon ngu (`vi`, 199 symbols).

## Luu y tai nguyen Kaggle

- Disk `/kaggle/working` gioi han ~19.5 GB: notebook mac dinh bat `--save-top-k 3` va tao symlink cho file WAV da chuan 22,050 Hz mono PCM_16.
- Neu het thoi gian phien GPU (~9-12h), luu checkpoint tu `/kaggle/working/piper-plus-vi-run/output/checkpoints/` thanh Kaggle Dataset moi va dien duong dan vao `RESUME_CHECKPOINT` o phien tiep theo.

In [ ]:
# ============================================================================
# CELL 1: CAU HINH DU AN VA THAM SO HUAN LUYEN
# ============================================================================
import os
from pathlib import Path


# --- Du lieu dau vao (Kaggle Input) ---
RAW_WAVS_DIR = "/kaggle/input/datasets/fablenguyen2messi/ngochuyen1/wavs"
RAW_METADATA_CSV = "/kaggle/input/datasets/fablenguyen2messi/ngochuyen1/metadata.csv"
SPEAKER_NAME = "ngochuyen"

# --- Nguon ma nguon piper-plus da tich hop Tieng Viet ---
# Cach 1 (khuyen nghi neu chua push len GitHub): nen thu muc PiperPlus thanh zip/dataset
#   va gan duong dan Kaggle Input vao day, vi du: "/kaggle/input/piper-plus-vi"
PIPER_REPO_LOCAL_OR_DATASET = ""
# Cach 2: Clone tu GitHub fork cua ban co chua nhanh Tieng Viet
PIPER_REPO_URL = "https://github.com/18200245/piper-plus-vietnamese.git"
PIPER_REPO_BRANCH = "dev"

WORK_DIR = "/kaggle/working/piper-plus-vi-run"
REPO_DIR = "/kaggle/working/piper-plus"

# --- Che do huan luyen ---
# Chon "finetune_zs_v1" (khuyen nghi) hoac "scratch"
TRAIN_MODE = "finetune_zs_v1"
# De tiep tuc phien Kaggle truoc, dien duong dan file .ckpt vao day (de trong neu chay moi)
RESUME_CHECKPOINT = ""

# --- Tham so audio & model ---
TARGET_SR = 22050
QUALITY = "medium"
MIN_DURATION_SEC = 0.25
MAX_DURATION_SEC = 20.0
BATCH_SIZE = 8
MAX_EPOCHS = 500 if TRAIN_MODE == "finetune_zs_v1" else 1000
LEARNING_RATE = 5e-5 if TRAIN_MODE == "finetune_zs_v1" else 2e-4
# "32-true" an toan nhat tren Kaggle T4/P100; chi dung "bf16-mixed" tren GPU Ampere/Hopper
PRECISION = "32-true"
CHECKPOINT_EPOCHS = 25
SAVE_TOP_K = 3
VAL_EVERY_N_EPOCHS = 25
AUDIO_LOG_EPOCHS = 25
EMA_DECAY = 0.9995
UNIFY_EMB_LANG = True
WANDB_API_KEY = ""

# --- Base checkpoint ayousanz/piper-plus-base (releases/zs-v1) ---
BASE_MODEL_REPO = "ayousanz/piper-plus-base"
BASE_REVISION = "3620ed788667cb76f08bd6cf2db8152c1f4c8bd1"
BASE_RELEASE = "releases/zs-v1"
BASE_CKPT_SHA256 = "86d0362241c2a3a898d23b9c4bce416ca97725024075fb844ae630e7f9b121f0"
BASE_CFG_SHA256 = "d085681edaa5029c54ea8329f3f23ed347ba46ca15cd80e766de9a6a643dafc0"

if TRAIN_MODE not in ("finetune_zs_v1", "scratch"):
    raise ValueError(f"TRAIN_MODE khong hop le: {TRAIN_MODE!r}")

os.makedirs(WORK_DIR, exist_ok=True)
print(f"[INFO] TRAIN_MODE={TRAIN_MODE}, MAX_EPOCHS={MAX_EPOCHS}, LR={LEARNING_RATE}, PRECISION={PRECISION}")

In [ ]:
# ============================================================================
# CELL 2: KIEM TRA GPU, DUNG LUONG DISK VA DATASET DAU VAO
# ============================================================================
import shutil
import subprocess
import torch


subprocess.run(["nvidia-smi"], check=False)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU 0           : {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    print(f"GPU count       : {torch.cuda.device_count()}")

usage = shutil.disk_usage("/kaggle/working")
print(f"Disk free       : {usage.free / 1024**3:.2f} GB / {usage.total / 1024**3:.2f} GB")

assert Path(RAW_WAVS_DIR).is_dir(), f"Khong tim thay thu muc WAV: {RAW_WAVS_DIR}"
assert Path(RAW_METADATA_CSV).is_file(), f"Khong tim thay file metadata: {RAW_METADATA_CSV}"
print(f"[OK] Tim thay du lieu dau vao cho speaker '{SPEAKER_NAME}'")

## 1. Cai dat ma nguon PiperPlus (nhanh Tieng Viet) va cac thu vien can thiet

Khong cai de `torch`/`torchaudio` san co cua Kaggle de giu nguyen CUDA runtime. Bien dich Cython `monotonic_align` va cai editable `piper-plus-g2p`, `piper-train`, `piper-plus` voi `--no-deps`.

In [ ]:
# ============================================================================
# CELL 3: NAP REPO PIPER-PLUS VA CAI DAT DEPENDENCIES
# ============================================================================
import shutil
import subprocess
import sys
import tarfile
import zipfile


repo_path = Path(REPO_DIR)
vi_marker = repo_path / "src" / "python" / "g2p" / "piper_plus_g2p" / "vietnamese.py"

if not vi_marker.is_file():
    if PIPER_REPO_LOCAL_OR_DATASET:
        src = Path(PIPER_REPO_LOCAL_OR_DATASET)
        if not src.exists():
            raise FileNotFoundError(f"Khong tim thay PIPER_REPO_LOCAL_OR_DATASET: {src}")
        if repo_path.exists():
            shutil.rmtree(repo_path)
        if src.is_dir():
            # Ho tro ca truong hop Kaggle Dataset chua truc tiep ma nguon hoac chua 1 thu muc con/file zip
            if (src / "src" / "python" / "g2p" / "piper_plus_g2p" / "vietnamese.py").is_file():
                shutil.copytree(src, repo_path)
            else:
                archives = list(src.glob("*.zip")) + list(src.glob("*.tar.gz")) + list(src.glob("*.tgz"))
                if archives:
                    src = archives[0]
                else:
                    nested = [d for d in src.iterdir() if (d / "src" / "python" / "g2p" / "piper_plus_g2p" / "vietnamese.py").is_file()]
                    if not nested:
                        raise FileNotFoundError(f"Thu muc {src} khong chua piper-plus ban Tieng Viet")
                    shutil.copytree(nested[0], repo_path)
        if src.is_file():
            tmp_extract = Path(WORK_DIR) / "_repo_extract"
            if tmp_extract.exists():
                shutil.rmtree(tmp_extract)
            tmp_extract.mkdir(parents=True, exist_ok=True)
            if src.suffix == ".zip":
                with zipfile.ZipFile(src, "r") as zf:
                    zf.extractall(tmp_extract)
            else:
                with tarfile.open(src, "r:*") as tf:
                    tf.extractall(tmp_extract)
            if (tmp_extract / "src" / "python" / "g2p" / "piper_plus_g2p" / "vietnamese.py").is_file():
                shutil.move(str(tmp_extract), str(repo_path))
            else:
                roots = list(tmp_extract.rglob("vietnamese.py"))
                matched = [r.parents[4] for r in roots if r.parts[-5:] == ("src", "python", "g2p", "piper_plus_g2p", "vietnamese.py")]
                if not matched:
                    raise FileNotFoundError(f"Archive {src} khong chua piper_plus_g2p/vietnamese.py")
                shutil.move(str(matched[0]), str(repo_path))
                shutil.rmtree(tmp_extract, ignore_errors=True)
    elif PIPER_REPO_URL:
        if repo_path.exists():
            shutil.rmtree(repo_path)
        subprocess.run(
            ["git", "clone", "--branch", PIPER_REPO_BRANCH, "--depth", "1", PIPER_REPO_URL, str(repo_path)],
            check=True,
        )
    else:
        raise RuntimeError(
            "Chua cau hinh nguon repo piper-plus co Tieng Viet. "
            "Vui long upload workspace nay thanh Kaggle Dataset va dien PIPER_REPO_LOCAL_OR_DATASET, "
            "hoac push len GitHub fork va dien PIPER_REPO_URL o Cell 1."
        )

assert vi_marker.is_file(), f"Thieu file Tieng Viet: {vi_marker}"
assert (repo_path / "scripts" / "prepare_vietnamese_finetune.py").is_file(), "Thieu prepare_vietnamese_finetune.py"
print(f"[OK] Repo san sang tai: {repo_path}")

deps = [
    "sea-g2p>=0.9.1",
    "soxr>=0.5.0",
    "soundfile>=0.12",
    "pytorch-lightning>=2.4.0",
    "torchmetrics>=1.4.0",
    "librosa>=0.10.0",
    "numba",
    "einops>=0.7",
    "onnx>=1.16.0",
    "onnxruntime>=1.18.0",
    "onnxscript",
    "onnxsim-prebuilt",
    "huggingface_hub>=0.23.0",
    "cython",
    "tensorboard",
    "tqdm",
    "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)

# Bien dich extension Cython monotonic_align ngay tai thu muc module
ma_dir = repo_path / "src" / "python" / "piper_train" / "vits" / "monotonic_align"
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=str(ma_dir), check=True)

# Cai editable cac package noi bo (--no-deps de giu nguyen PyTorch CUDA cua Kaggle)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(repo_path / "src" / "python" / "g2p"),
        "-e",
        str(repo_path / "src" / "python"),
        "-e",
        str(repo_path / "src" / "python_run"),
    ],
    check=True,
)

for p in [
    str(repo_path / "src" / "python" / "g2p"),
    str(repo_path / "src" / "python"),
    str(repo_path / "src" / "python_run"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import piper_plus_g2p
import piper_train
import sea_g2p
from piper_train.vits.monotonic_align import maximum_path

print(f"[OK] Cai dat thanh cong. Available G2P languages: {piper_plus_g2p.available_languages()}")

## 2. Kiem tra va chuan hoa du lieu (`file.wav|ngochuyen|text`)

Doc `/kaggle/input/datasets/fablenguyen2messi/ngochuyen1/metadata.csv`, kiem tra file audio trong `wavs/`, loc bo mau loi/qua ngan/qua dai, va chuan hoa ve 22,050 Hz mono PCM_16 trong `WORK_DIR/dataset_22k` (su dung symlink neu file goc da dat chuan de tiet kiem dung luong Kaggle).

In [ ]:
# ============================================================================
# CELL 4: KIEM TRA METADATA VA CHUAN HOA AUDIO 22,050 HZ MONO
# ============================================================================
import numpy as np
import soundfile as sf
import soxr
from tqdm.auto import tqdm


raw_wavs_root = Path(RAW_WAVS_DIR)
raw_meta_path = Path(RAW_METADATA_CSV)
STAGED_DATASET_DIR = Path(WORK_DIR) / "dataset_22k"
STAGED_WAVS_DIR = STAGED_DATASET_DIR / "wavs"
if STAGED_DATASET_DIR.exists():
    shutil.rmtree(STAGED_DATASET_DIR)
STAGED_WAVS_DIR.mkdir(parents=True, exist_ok=True)

# Tao chi muc nhanh phong truong hop metadata chua ten file hoac ten khong co .wav
wav_by_name = {}
wav_by_stem = {}
for p in raw_wavs_root.rglob("*"):
    if p.is_file() and p.suffix.lower() in (".wav", ".flac", ".ogg"):
        wav_by_name.setdefault(p.name, p)
        wav_by_stem.setdefault(p.stem, p)

raw_lines = raw_meta_path.read_text(encoding="utf-8-sig").splitlines()
parsed_rows = []
skipped_format = 0

for idx, line in enumerate(raw_lines):
    line = line.strip()
    if not line:
        continue
    parts = line.split("|")
    if len(parts) < 2:
        skipped_format += 1
        continue
    fname = parts[0].strip()
    # Tu dong bo qua dong tieu de CSV neu ton tai va khong trung ten file audio that
    if idx == 0 and fname.lower() in ("file", "filename", "wav", "audio", "path", "id"):
        if not (raw_wavs_root / fname).exists() and fname not in wav_by_stem:
            continue
    if len(parts) >= 3:
        spk = parts[1].strip()
        txt = "|".join(parts[2:]).strip()
    else:
        spk = SPEAKER_NAME
        txt = parts[1].strip()
    if not txt:
        skipped_format += 1
        continue
    parsed_rows.append((fname, spk, txt))

stats = {
    "missing": 0,
    "corrupt": 0,
    "too_short": 0,
    "too_long": 0,
    "symlinked": 0,
    "resampled": 0,
}
durations = []
staged_rows = []
used_names = set()
speakers_seen = set()

for idx, (fname, spk, txt) in enumerate(tqdm(parsed_rows, desc="Validating audio")):
    candidates = [
        raw_wavs_root / fname,
        raw_wavs_root / f"{fname}.wav",
        raw_meta_path.parent / fname,
        wav_by_name.get(Path(fname).name),
        wav_by_stem.get(Path(fname).stem),
    ]
    src_wav = next((c for c in candidates if c is not None and c.is_file()), None)
    if src_wav is None or src_wav.stat().st_size == 0:
        stats["missing"] += 1
        continue

    try:
        info = sf.info(str(src_wav))
        dur = info.frames / float(info.samplerate) if info.samplerate > 0 else 0.0
    except Exception:
        stats["corrupt"] += 1
        continue

    if dur < MIN_DURATION_SEC:
        stats["too_short"] += 1
        continue
    if dur > MAX_DURATION_SEC:
        stats["too_long"] += 1
        continue

    out_name = f"{src_wav.stem}.wav"
    if out_name in used_names:
        out_name = f"{idx:06d}_{src_wav.stem}.wav"
    used_names.add(out_name)
    dst_wav = STAGED_WAVS_DIR / out_name

    if (
        src_wav.suffix.lower() == ".wav"
        and info.samplerate == TARGET_SR
        and info.channels == 1
        and info.subtype == "PCM_16"
    ):
        os.symlink(src_wav, dst_wav)
        stats["symlinked"] += 1
    else:
        audio, sr = sf.read(str(src_wav), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = soxr.resample(audio, sr, TARGET_SR, quality="HQ")
        peak = float(np.abs(audio).max()) if audio.size else 0.0
        if peak > 0.99:
            audio = audio / peak * 0.95
        sf.write(str(dst_wav), audio, TARGET_SR, subtype="PCM_16")
        stats["resampled"] += 1

    # Thong nhat dau gach dung neu text co ky tu '|' de ljspeech_dataset doc row[-1] chuan xac
    clean_txt = " ".join(txt.replace("|", ", ").split())
    staged_rows.append(f"{out_name}|{SPEAKER_NAME}|{clean_txt}")
    durations.append(dur)
    speakers_seen.add(spk)

if not staged_rows:
    raise RuntimeError(f"Khong co mau hop le nao! Thong ke loi: {stats}")

staged_meta = STAGED_DATASET_DIR / "metadata.csv"
staged_meta.write_text("\n".join(staged_rows) + "\n", encoding="utf-8")

dur_arr = np.array(durations)
print("=== THONG KE DATASET DA CHUAN HOA ===")
print(f"Tong so cau hop le : {len(staged_rows)} (speakers trong file goc: {sorted(speakers_seen)})")
print(f"Tong thoi luong    : {dur_arr.sum() / 3600:.2f} gio ({dur_arr.sum() / 60:.1f} phut)")
print(f"Thoi luong (s)     : min={dur_arr.min():.2f}s | median={np.median(dur_arr):.2f}s | max={dur_arr.max():.2f}s")
print(f"Symlinked / Resamp : {stats['symlinked']} / {stats['resampled']}")
print(f"Bo qua             : format={skipped_format}, missing={stats['missing']}, corrupt={stats['corrupt']}, short={stats['too_short']}, long={stats['too_long']}")
print("\n5 mau dau tien:")
for sample_line in staged_rows[:5]:
    print(" ", sample_line[:120])

## 3. Smoke Test G2P Tieng Viet (`sea-g2p` + `PiperEncoder` strict)

Kiem tra nhanh kha nang chuyen am va anh xa ID khong loi tren ca cau thu nghiem lan mot so cau thuc te tu `metadata.csv`.

In [ ]:
# ============================================================================
# CELL 5: SMOKE TEST VIETNAMESE G2P + PHONEME ENCODER
# ============================================================================
from piper_plus_g2p import get_phonemizer
from piper_plus_g2p.encode.encoder import PiperEncoder
from piper_plus_g2p.encode.id_maps import get_phoneme_id_map


vi_id_map = get_phoneme_id_map("vi")
vi_phonemizer = get_phonemizer("vi")
vi_encoder = PiperEncoder(vi_id_map, strict=True)

sample_texts = [
    "Xin chào, tôi là trợ lý giọng nói tiếng Việt.",
    "Ma, mà, má, mả, mã, mạ — sáu thanh điệu tiếng Việt.",
    "Hôm nay là ngày 27/11/2026, tôi dùng iPhone 15 Pro và GitHub.",
]
for line in staged_rows[:5]:
    sample_texts.append(line.split("|", 2)[-1])

for i, txt in enumerate(sample_texts):
    tokens, prosody = vi_phonemizer.phonemize_with_prosody(txt)
    ids, aligned_prosody = vi_encoder.encode_with_prosody(tokens, prosody)
    assert len(ids) == len(aligned_prosody), "Do dai phoneme_ids va prosody khong khop"
    if i < 4:
        print(f"[{i}] text='{txt[:70]}' -> tokens={len(tokens)}, ids={len(ids)}")

print(f"[OK] G2P smoke test thanh cong ({len(vi_id_map)} symbols, strict encoder = 0 unknown tokens)")

## 4. Tien xu ly du lieu (`piper_train.preprocess`)

Trich xuat spectrogram va `phoneme_ids` + `prosody_features` cho bo du lieu Tieng Viet.

In [ ]:
# ============================================================================
# CELL 6: CHAY PIPER_TRAIN.PREPROCESS (--language vi)
# ============================================================================
import json


PREPARED_DIR = Path(WORK_DIR) / "prepared_vi"
if PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)

num_workers = min(4, os.cpu_count() or 2)
subprocess.run(
    [
        sys.executable,
        "-m",
        "piper_train.preprocess",
        "--language",
        "vi",
        "--input-dir",
        str(STAGED_DATASET_DIR),
        "--output-dir",
        str(PREPARED_DIR),
        "--dataset-format",
        "ljspeech",
        "--single-speaker",
        "--sample-rate",
        str(TARGET_SR),
        "--max-workers",
        str(num_workers),
    ],
    cwd=str(REPO_DIR),
    check=True,
)

prep_cfg = json.loads((PREPARED_DIR / "config.json").read_text(encoding="utf-8"))
prep_utts = sum(1 for _ in (PREPARED_DIR / "dataset.jsonl").read_text(encoding="utf-8").splitlines())
print(f"[OK] Tien xu ly xong {prep_utts} mau")
print(
    f"num_symbols={prep_cfg['num_symbols']}, num_speakers={prep_cfg['num_speakers']}, "
    f"language_id_map={prep_cfg.get('language_id_map')}, "
    f"prosody_num_symbols={prep_cfg.get('prosody_num_symbols')}"
)

## 5. Tai base checkpoint `zs-v1` va dong bo ID cho Fine-Tuning

Buoc nay chi chay khi `TRAIN_MODE == "finetune_zs_v1"`: tai `base.ckpt` va `config.json` tu `ayousanz/piper-plus-base`, kiem tra SHA-256, va chay `scripts/prepare_vietnamese_finetune.py` de giu nguyen ID `0..172` va noi them token Tieng Viet tu ID `173..198` cung voi `language_id` `"vi": 6`.

In [ ]:
# ============================================================================
# CELL 7: TAI BASE MODEL ZS-V1 VA CAN CHINH DATASET
# ============================================================================
import hashlib
from huggingface_hub import hf_hub_download


def download_verified(filename: str, expected_sha256: str, out_dir: Path) -> Path:
    path = Path(
        hf_hub_download(
            repo_id=BASE_MODEL_REPO,
            revision=BASE_REVISION,
            filename=f"{BASE_RELEASE}/{filename}",
            local_dir=str(out_dir),
            token=False,
        )
    )
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f"SHA256 mismatch cho {filename}: {actual} != {expected_sha256}")
    return path


BASE_CHECKPOINT = None
if TRAIN_MODE == "finetune_zs_v1":
    base_dir = Path(WORK_DIR) / "base_zs_v1"
    base_dir.mkdir(parents=True, exist_ok=True)
    BASE_CHECKPOINT = download_verified("base.ckpt", BASE_CKPT_SHA256, base_dir)
    BASE_CONFIG = download_verified("config.json", BASE_CFG_SHA256, base_dir)

    ALIGNED_DIR = Path(WORK_DIR) / "aligned_vi_zs_v1"
    if ALIGNED_DIR.exists():
        shutil.rmtree(ALIGNED_DIR)

    subprocess.run(
        [
            sys.executable,
            str(Path(REPO_DIR) / "scripts" / "prepare_vietnamese_finetune.py"),
            "--input-dir",
            str(PREPARED_DIR),
            "--base-config",
            str(BASE_CONFIG),
            "--output-dir",
            str(ALIGNED_DIR),
        ],
        cwd=str(REPO_DIR),
        check=True,
    )
    TRAIN_DATASET_DIR = ALIGNED_DIR
else:
    TRAIN_DATASET_DIR = PREPARED_DIR

train_cfg = json.loads((TRAIN_DATASET_DIR / "config.json").read_text(encoding="utf-8"))
print(f"[OK] TRAIN_DATASET_DIR = {TRAIN_DATASET_DIR}")
print(
    f"num_symbols={train_cfg['num_symbols']}, num_languages={train_cfg.get('num_languages', 1)}, "
    f"language_id_map={train_cfg.get('language_id_map')}, pua_compat_version={train_cfg.get('pua_compat_version')}"
)

## 6. Huan luyen Model (`piper_train`)

Khi `TRAIN_MODE == "finetune_zs_v1"`, lenh train tu dong bat `--expand-embeddings --no-freeze-dp --prosody-language-ids 0,6` va giu `--save-top-k 3` de bao ve o dia Kaggle.

In [ ]:
# ============================================================================
# CELL 8: HUAN LUYEN MODEL
# ============================================================================
OUTPUT_DIR = Path(WORK_DIR) / f"output_{SPEAKER_NAME}_{TRAIN_MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_env = os.environ.copy()
train_env["MPLBACKEND"] = "Agg"
if WANDB_API_KEY:
    train_env["WANDB_API_KEY"] = WANDB_API_KEY
else:
    train_env["WANDB_MODE"] = "disabled"

cmd = [
    sys.executable,
    "-m",
    "piper_train",
    "--dataset-dir",
    str(TRAIN_DATASET_DIR),
    "--prosody-dim",
    "16",
    "--accelerator",
    "gpu",
    "--devices",
    "1",
    "--precision",
    PRECISION,
    "--max_epochs",
    str(MAX_EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--checkpoint-epochs",
    str(CHECKPOINT_EPOCHS),
    "--save-top-k",
    str(SAVE_TOP_K),
    "--quality",
    QUALITY,
    "--base_lr",
    str(LEARNING_RATE),
    "--disable_auto_lr_scaling",
    "--ema-decay",
    str(EMA_DECAY),
    "--max-phoneme-ids",
    "400",
    "--no-wavlm",
    "--val-every-n-epochs",
    str(VAL_EVERY_N_EPOCHS),
    "--audio-log-epochs",
    str(AUDIO_LOG_EPOCHS),
    "--default_root_dir",
    str(OUTPUT_DIR),
]

if RESUME_CHECKPOINT:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(f"Khong tim thay RESUME_CHECKPOINT: {RESUME_CHECKPOINT}")
    cmd.extend(["--resume_from_checkpoint", RESUME_CHECKPOINT])
    if TRAIN_MODE == "finetune_zs_v1":
        cmd.extend(["--prosody-language-ids", "0,6"])
elif TRAIN_MODE == "finetune_zs_v1":
    cmd.extend([
        "--resume-from-multispeaker-checkpoint",
        str(BASE_CHECKPOINT),
        "--expand-embeddings",
        "--no-freeze-dp",
        "--prosody-language-ids",
        "0,6",
    ])

print("[INFO] Bat dau huan luyen:")
print(" ", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), env=train_env, check=True)

## 7. Xuat ONNX & sao chep `config.json`

Khi export tu mo hinh `finetune_zs_v1` (7 ngon ngu, don nguoi noi), **bat buoc** truyen `--unify-emb-lang-source 6` de copy embedding cua Tieng Viet (`vi = 6`) thay vi lay mac dinh `0` (Tieng Nhat).

In [ ]:
# ============================================================================
# CELL 9: TIM CHECKPOINT MOI NHAT VA XUAT MODEL ONNX
# ============================================================================
import re


ckpt_candidates = list((OUTPUT_DIR / "checkpoints").glob("epoch=*.ckpt"))
ckpt_candidates.extend(OUTPUT_DIR.glob("lightning_logs/version_*/checkpoints/epoch=*.ckpt"))


def _ckpt_sort_key(path: Path) -> tuple[int, int]:
    ep = re.search(r"epoch=([0-9]+)", path.name)
    st = re.search(r"step=([0-9]+)", path.name)
    return (int(ep.group(1)) if ep else -1, int(st.group(1)) if st else -1)


if ckpt_candidates:
    FINAL_CHECKPOINT = sorted(set(ckpt_candidates), key=_ckpt_sort_key)[-1]
elif (OUTPUT_DIR / "checkpoints" / "last.ckpt").is_file():
    FINAL_CHECKPOINT = OUTPUT_DIR / "checkpoints" / "last.ckpt"
else:
    raise FileNotFoundError(f"Khong tim thay checkpoint nao trong {OUTPUT_DIR}")

ONNX_OUTPUT = OUTPUT_DIR / f"{SPEAKER_NAME}-vi.onnx"
ONNX_CONFIG = Path(f"{ONNX_OUTPUT}.json")

export_cmd = [
    sys.executable,
    "-m",
    "piper_train.export_onnx",
    str(FINAL_CHECKPOINT),
    str(ONNX_OUTPUT),
]
num_langs = int(train_cfg.get("num_languages", 1))
if num_langs > 1:
    if UNIFY_EMB_LANG:
        vi_lid = int(train_cfg.get("language_id_map", {}).get("vi", 0))
        export_cmd.extend(["--unify-emb-lang", "--unify-emb-lang-source", str(vi_lid)])
    else:
        export_cmd.append("--no-unify-emb-lang")

export_env = os.environ.copy()
export_env["CUDA_VISIBLE_DEVICES"] = ""
subprocess.run(export_cmd, cwd=str(REPO_DIR), env=export_env, check=True)
shutil.copy2(TRAIN_DATASET_DIR / "config.json", ONNX_CONFIG)

print(f"[OK] Checkpoint  : {FINAL_CHECKPOINT}")
print(f"[OK] ONNX model  : {ONNX_OUTPUT} ({ONNX_OUTPUT.stat().st_size / 1024**2:.1f} MB)")
print(f"[OK] ONNX config : {ONNX_CONFIG}")

## 8. Chay thu suy luan ONNX va nghe thu am thanh

Sinh thu 4 cau gom Tieng Viet chuan, cap thanh dieu, so/ngay thang, va chen tu Tieng Anh.

In [ ]:
# ============================================================================
# CELL 10: INFERENCE THU NGHIEM VOI ONNX VA NGHE AUDIO
# ============================================================================
from IPython.display import Audio, display


INFER_DIR = OUTPUT_DIR / "test_audio"
INFER_DIR.mkdir(parents=True, exist_ok=True)

test_sentences = [
    "Xin chào các bạn, hôm nay trời Hà Nội thật đẹp và mát mẻ.",
    "Ma, mà, má, mả, mã, mạ là sáu thanh điệu cơ bản của tiếng Việt.",
    "Ngày 27/11/2026, đơn hàng trị giá 500.000đ được giảm 20%.",
    "Tôi đang dùng iPhone 15 Pro để kiểm tra chất lượng livestream.",
]

for i, sentence in enumerate(test_sentences):
    sample_dir = INFER_DIR / f"sample_{i}"
    sample_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "piper_train.infer_onnx",
            "--model",
            str(ONNX_OUTPUT),
            "--config",
            str(ONNX_CONFIG),
            "--language",
            "vi",
            "--device",
            "cpu",
            "--output-dir",
            str(sample_dir),
            "--text",
            sentence,
        ],
        cwd=str(REPO_DIR),
        check=True,
    )
    wav_out = sample_dir / "output.wav"
    final_wav = INFER_DIR / f"ngochuyen_sample_{i}.wav"
    shutil.move(str(wav_out), str(final_wav))
    print(f"Sample {i}: {sentence}")
    display(Audio(str(final_wav), rate=TARGET_SR))

## 9. Dong goi ket qua de tai ve tu Kaggle Output

Sao chep `.onnx`, `.onnx.json`, file am thanh mau va tao file nen `/kaggle/working/ngochuyen-vi-onnx.zip`.

In [ ]:
# ============================================================================
# CELL 11: DONG GOI FILE XUAT DE TAI VE
# ============================================================================
RELEASE_DIR = Path("/kaggle/working/release")
if RELEASE_DIR.exists():
    shutil.rmtree(RELEASE_DIR)
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(ONNX_OUTPUT, RELEASE_DIR / ONNX_OUTPUT.name)
shutil.copy2(ONNX_CONFIG, RELEASE_DIR / ONNX_CONFIG.name)
for wav_file in sorted(INFER_DIR.glob("ngochuyen_sample_*.wav")):
    shutil.copy2(wav_file, RELEASE_DIR / wav_file.name)

zip_path = shutil.make_archive("/kaggle/working/ngochuyen-vi-onnx", "zip", root_dir=str(RELEASE_DIR))
print(f"[OK] Release dir : {RELEASE_DIR}")
print(f"[OK] Zip archive : {zip_path} ({Path(zip_path).stat().st_size / 1024**2:.1f} MB)")
print(f"[INFO] Checkpoint gan nhat (de resume neu can): {FINAL_CHECKPOINT}")